# Image-grounded bug report verification

**Recipe 93** · Level 3 (Intermediate) · Decision type: `Noul`

Check whether a screenshot supports two claimed save symptoms alongside error logs and expected behavior.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)


## What you will build

Build a bounded evidence checker for reports that a draft save shows an error and lacks the expected Saved confirmation. Four independent `Noul` propositions separate those visual symptoms from log evidence and page identity. Python combines the answers with exact artifact checks and preserves evidence in a simulated human [review](../../docs/glossary.md#review) queue: review means a person must decide, and is never an answered result.

All screenshots are synthetic application drawings authored with Pillow, not captured product incidents. Supporting a visible symptom does not prove a defect's cause, persistence, reproducibility, or exploitability. No report text is executed, no browser is controlled, and no external ticket changes.

## Setup and run mode

The committed [run mode](../../docs/glossary.md#run-mode) is synthetic replay: hand-written typed answers exercise the pipeline without inference. There are 12 validation reports, 12 held-out test reports and two unscored demonstrations, with one 480 × 360 image and four questions per request. Every independent question shares that one image in one batch. A cached session permits at most 26 unique requests; repeats reuse the result.

The same questions can use experimental image inference with pinned `jev-1.14.0` via `images=True` ([S09](https://docs.typesafe.ai/images), [S11](https://docs.typesafe.ai/models), [S12](https://typesafe.ai/blog/jev-1-14)). Long context is unnecessary here. Live exploration is unreportable; `capture.py` supplies a separate, opt-in recording path, and the copied notebook can replay actual recorded answers.

In [ ]:
import json
from pathlib import Path

from IPython.display import display
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import accuracy, confusion_matrix, evaluate_outcomes
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

apply_style()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
session = helpers.RequestSession(backend)
scored = [e for e in examples if e.split in ("validation", "test")]
results = {}
requests = {}
offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
selection = " (selection step)"
reportable = backend.mode != "live"
run_header(93, "Image-grounded bug report verification", backend=backend, n_examples=len(scored))

## The state

The image, bug description, expected page and confirmation, log events, interaction metadata and capture manifest are legitimate evidence. Python omits fixture identifiers, split membership, rendering instructions and labels. An opaque, content-addressed image reference binds the actual PNG bytes to the [replay key](../../docs/glossary.md#replay-key), the hash used to retrieve this exact stored request.

The capture manifest is an assumed trusted input, not an authentication system. Python checks matching build and run identifiers, a screenshot taken 0–5 seconds after the interaction, and logs within 1 second of it. Those checks establish association only; a forged manifest could still pass. A separate visual question detects the wrong page even when metadata agrees.

In [ ]:
demo = [e for e in examples if e.split == "demo"]
example = demo[0]
request = helpers.build_request(example.fields)
requests[example.id] = request
state, questions, assets = request
print(json.dumps(state, indent=2))
with Image.open(example.fields["screenshot"]) as image:
    display(image.copy())

## The questions

A `Noul` is a probability that one stated proposition is true ([S02](https://docs.typesafe.ai/primitives)). `error_visible` and `saved_missing` address the two report symptoms separately. The latter requires a visible confirmation area: obscuration is not evidence of absence. `log_failure` asks about explicit log support, and `view_matches` asks about the pictured page. No answer is fed into another question, so all four belong in one request.

Report text and image text are untrusted evidence. The propositions explicitly refuse to treat them as instructions. That boundary reduces ambiguity in the task; it is not a guarantee that an inferred answer resists malicious text.

In [ ]:
for name, question in questions.items():
    print(name, question.to_dict())

## One answer up close

These are stored, synthetic probabilities of yes. `Noul` has no API confidence field. The derived certainty `abs(2*p - 1)` is 0 at a tie and 1 at either extreme; it does not establish calibration. The cookbook's [confidence gate](../../docs/glossary.md#confidence-gate) is the Python rule that declines a judgment below a selected certainty threshold ([S03](https://docs.typesafe.ai/confidence)).

In [ ]:
result = session.decide(request)
results[example.id] = result
for name, answer in result.answers.items():
    print(name)
    show_answer(answer)

## Python's part

Python owns three outcomes: `supported`, `not_supported`, and `review`. A supported report requires both visual symptoms, a failed-save log, the correct page and matching artifacts. A negative result requires visible Saved confirmation, no failure indicator, and a success interpretation of the log. A screenshot cannot disprove an intermittent bug; `not_supported` describes only the supplied evidence.

An invalid answer, a tie, an artifact mismatch, the wrong page, or conflicting evidence always goes to review. A missing marker alone also goes to review. Every accepted result must clear the same gate; there is no low-certainty fallback exemption. The following contrast uses a provisional demo-only threshold of 0.8; the evaluated threshold is chosen later on validation.

`ActionLog` records simulated annotations with `executed=False`. `ReviewQueue` retains the image reference and all four typed answers. Neither container changes a ticket, runs an exploit, or declares a root cause.

In [ ]:
demo_queue, demo_log = ReviewQueue(), ActionLog()
for e in demo:
    requests[e.id] = helpers.build_request(e.fields)
    results[e.id] = session.decide(requests[e.id])
    routing = helpers.route(
        e.id, e.fields, results[e.id].answers, 0.8, requests[e.id][0]["screenshot"]
    )
    helpers.dispatch(routing, results[e.id], demo_queue, demo_log)
    print(e.id, routing.outcome, routing.reason)
print("Simulated annotation:", demo_log.to_dicts()[0])
print("Pending review:", demo_queue.to_dicts()[0])

## Evaluation

A [gold label](../../docs/glossary.md#gold-label) is the author's independent expected judgment, written in `labels.jsonl`: four visible/text claim labels plus the report disposition. Labels are never request input. All uncertainty here is represented in the answers and review policy; the synthetic drawings themselves have author-known labels.

Select the lowest certainty threshold attaining at least 95% pooled claim [accuracy](../../docs/glossary.md#accuracy), the fraction of correct judgments, on validation only. Candidates are the distinct observed validation claim certainties. Pooling gives four claims per report; their dependence makes this a pipeline check, not 48 independent trials or a statistical guarantee. This objective is about claim acceptance, not final report acceptance. Freeze the resulting scalar before obtaining test answers.

Report [coverage](../../docs/glossary.md#coverage), the fraction of reports actually answered, and [risk](../../docs/glossary.md#risk), the wrong fraction among answered reports, from `evaluate_outcomes` using the routing rule's real accept flags. Review is abstention. Report total disposition accuracy separately so a correct review cannot inflate coverage.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
for e in validation:
    requests[e.id] = helpers.build_request(e.fields)
    results[e.id] = session.decide(requests[e.id])
threshold = helpers.select_policy(validation, labels, results)
val_correct, val_certainty = helpers.claim_pool(validation, labels, results)
val_claims = evaluate_outcomes([c >= threshold for c in val_certainty], val_correct)
print(
    f"Validation N={len(validation)} reports, N={len(val_correct)} claims: frozen threshold={threshold:.4f}{selection}{check}"
)
print(
    f"Validation N={val_claims.n_total} claims: coverage={val_claims.coverage:.4f}, accuracy={val_claims.accuracy:.4f}{selection}{check}"
)

### A text-and-pixel baseline

The deterministic baseline reads the same screenshot bytes and log events. It detects a red banner in a fixed rectangle and a green Saved marker in another, then checks `SAVE_FAILED` in the logs. It uses the same artifact and contradiction rules, but assumes the page title matches because it has no OCR. The color thresholds and viewport are fixed before validation: red fraction > 0.20 and green fraction < 0.15. No baseline setting reads labels, fixture names or renderer keys.

This is useful for a stable UI layout but fails on an amber error, a red help example and a different page with the same controls. Those limitations are exercised on both splits. The baseline and Noul pipeline see identical test reports; neither gets a special subset.

In [ ]:
test = [e for e in examples if e.split == "test"]
for e in test:
    requests[e.id] = helpers.build_request(e.fields)
    results[e.id] = session.decide(requests[e.id])
queue, action_log = ReviewQueue(), ActionLog()
routings = []
for e in test:
    routing = helpers.route(
        e.id, e.fields, results[e.id].answers, threshold, requests[e.id][0]["screenshot"]
    )
    routings.append(routing)
    helpers.dispatch(routing, results[e.id], queue, action_log)
gold = [labels[e.id]["outcome"] for e in test]
predicted = [r.outcome for r in routings]
baseline_predicted = [helpers.baseline(e.fields)[0] for e in test]
if reportable:
    for name, outcomes in (
        ("Noul pipeline", predicted),
        ("Text+pixel baseline", baseline_predicted),
    ):
        accepted = [outcome != helpers.REVIEW for outcome in outcomes]
        correct = [actual == expected for actual, expected in zip(outcomes, gold, strict=True)]
        metrics = evaluate_outcomes(accepted, correct)
        print(
            f"Test {name}: N={metrics.n_total}, answered={metrics.n_answered}, review={metrics.n_total - metrics.n_answered}{check}"
        )
        print(
            f"Test {name}: N={metrics.n_total}, coverage={metrics.coverage:.4f}, abstention={1 - metrics.coverage:.4f}{check}"
        )
        print(
            f"Test {name}: N_answered={metrics.n_answered}, accuracy={metrics.accuracy:.4f}, risk={metrics.risk:.4f}{check}"
        )
        print(
            f"Test {name}: N={len(test)}, disposition accuracy={accuracy(gold, outcomes):.4f}{check}"
        )
    claim_correct, _claim_certainty = helpers.claim_pool(test, labels, results)
    print(
        f"Test raw claim accuracy: N={len(claim_correct)}, accuracy={sum(claim_correct) / len(claim_correct):.4f}{check}"
    )
else:
    print(
        "Live exploration is unrecorded: no held-out metrics reported. Use capture.py, then replay."
    )
print("Simulated test annotations:", len(action_log), "pending human reviews:", len(queue))

In [ ]:
if reportable:
    matrix = confusion_matrix(gold, predicted, labels=["supported", "not_supported", "review"])
    print(f"Test N={len(test)} gold rows / predicted columns {list(matrix.labels)}{check}")
    print(f"Counts: {matrix.matrix.tolist()}{check}")
    display(plot_confusion_matrix(matrix, title=f"Test dispositions, N={len(test)}{check}"))

### Inspect the accepted mistake

The held-out wrong-page example has matching build, run and time metadata, but visibly shows Billing settings. Its synthetic `view_matches` answer is confidently wrong, so the frozen policy accepts a supported report where the label requires review. This is an explicit residual failure: deterministic checks cannot repair an erroneous visual judgment when the other evidence agrees. The test evidence below is inspected after reporting; it does not retune the policy.

In [ ]:
if reportable:
    wrong = [
        (e, r)
        for e, r in zip(test, routings, strict=True)
        if r.accepted and r.outcome != labels[e.id]["outcome"]
    ]
    print(f"Accepted wrong reports: {len(wrong)} of N={len(test)}{check}")
    for e, r in wrong:
        print(
            e.id,
            "predicted",
            r.outcome,
            "gold",
            labels[e.id]["outcome"],
            "certainty",
            round(r.certainty, 4),
        )
        show_answer(results[e.id].answers["view_matches"])
        with Image.open(e.fields["screenshot"]) as image:
            display(image.copy())

## What was and was not measured

Not measured live: the committed run replays synthetic answers, so its numbers check the pipeline and say nothing about how Jev performs. N is printed below for reports, individual claims, split sizes and requests. The two demos never enter aggregate metrics.

For recorded replay, the code prints the actual returned model and capture dates; optional `measurements.json` supplies measured call durations from capture. Usage absent from the API stays null. Synthetic model, token counts, inference latency and cost are null. Replay execution time is never treated as inference latency; no billing estimate is fabricated.

The small authored set covers only one 480 × 360 UI, two named symptoms and one log event per report. It does not measure OCR robustness, other layouts, causal diagnosis, exploit execution, or adversarial security. The four judgments are dependent evidence about a shared screenshot, not independent confirmations.

In [ ]:
print(
    f"Provenance: {backend.mode}; N={len(scored)} scored reports, {len(scored) * 4} claims{check}"
)
print(
    f"N_validation={len(validation)}, N_test={len(test)}, N_demo={len(demo)}; requests={session.attempts}, images/request=1, questions/request=4{check}"
)
if offline:
    print("Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(scored)} scored reports; validation={len(validation)}, test={len(test)}{check}")
    print("Model=None, capture date=None.")
else:
    print(
        "Returned model:",
        backend.model,
        "capture dates:",
        list(getattr(backend, "recorded_dates", ())) or "unrecorded live exploration",
    )
measurements = (
    json.loads(Path("measurements.json").read_text()) if Path("measurements.json").exists() else {}
)
accounting = []
for e in test:
    captured = measurements.get(e.replay_keys[0], {}) if backend.mode == "recorded" else {}
    accounting.append(
        run_record(
            results[e.id],
            mode=backend.mode,
            measured_latency_seconds=captured.get("latency_seconds"),
        )
    )
for field in ("input_tokens", "output_tokens", "latency_seconds", "cost_usd"):
    values = [row[field] for row in accounting]
    total = sum(values) if all(value is not None for value in values) else None
    print(f"Test N={len(test)} requests: total {field}={total}{check}")

## Next steps

Collect consented, de-identified screenshots with trusted capture metadata and independently adjudicated labels. Add unreadable or obscured confirmation areas, different layouts, and recovered failures with multiple log events. Keep a new test set untouched while redesigning the policy; inspect claim-level and report-level errors separately.

See [Claim support classification](../20-claim-support-classification/) for textual evidence judgments, [Tool outcome verification](../29-tool-outcome-verification/) for simulated result checks, and [Abstention and review queues](../38-abstention-and-review-queues/) for coverage and review tradeoffs. The experimental image contract is documented in [S09](https://docs.typesafe.ai/images).